# Reading motor imagery from 59-channel EEG with CP tensor decomposition

### The idea in plain words

**EEG** records the brain's electrical activity with electrodes on the scalp. Here, 59
electrodes record while a person *imagines* moving their left hand, right hand, or foot
(**motor imagery**). Imagining a movement changes brain rhythms over the motor cortex, and a
brain-computer interface tries to detect which movement was imagined from a *single* short
recording (**single-trial decoding**). That is hard: EEG is noisy and the signals are tiny.

Our data is naturally a **tensor** (a block of numbers with more than two dimensions):

```
trials  x  channels  x  time
(200)      (59)         (300 samples)
```

**CP decomposition** (CANDECOMP/PARAFAC) writes that block as a sum of a few simple
"patterns". Each pattern is one **spatial map** (which electrodes), one **time course** (when),
and one **loading per trial** (how strongly that trial shows the pattern). With rank 4, the
whole 200 x 59 x 300 block is summarised by 4 patterns, and each trial by just 4 numbers.

**Question:** can those 4 numbers per trial tell us which movement the person imagined?

We keep it honest: CP is fitted on training trials only, test trials are *projected* onto
the learned maps, and accuracy is measured with cross-validation.

## 1. Setup

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorly"], check=True)

In [ ]:
import os, glob, json
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat                       # read MATLAB .mat files
from scipy.signal import butter, filtfilt          # band-pass filtering
import tensorly as tl
from tensorly.decomposition import parafac         # CP decomposition
from tensorly.tenalg import khatri_rao
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
rng = np.random.default_rng(SEED)
tl.set_backend("numpy")

## 2. Get the data

**Dataset:** BCI Competition IV, data set 1 (Berlin BCI group). 7 subjects (a to g), 59 EEG
channels, 100 Hz version, 200 calibration trials per subject, 2 classes per subject.

**How to download (free, two minutes):**
1. Go to https://www.bbci.de/competition/iv/ and scroll to *Download of data sets*.
2. Fill in the form (name, email). A download link arrives by email.
3. Download the **100 Hz MATLAB calibration** files: `BCICIV_calib_ds1a.mat` ... `BCICIV_calib_ds1g.mat`.
4. In Colab, click the folder icon on the left, then the upload icon, and upload the .mat
   files. (Or put them in your Google Drive and mount it.)

Subjects a, b, f, g are real recordings. c, d, e were later revealed to be artificially
generated, so we analyse the four real ones.

In [ ]:
# Look for the .mat files in a few likely places.
candidates = [os.environ.get("EEG_DIR", ""), ".", "/content", "/content/drive/MyDrive"]
mat_files = []
for d in candidates:
    if d and os.path.isdir(d):
        mat_files = sorted(glob.glob(os.path.join(d, "BCICIV_calib_ds1*.mat")))
        if mat_files:
            break

if not mat_files:
    raise FileNotFoundError("No BCICIV_calib_ds1*.mat files found. Upload them first (see the cell above).")

# keep only the real subjects (skip the artificial c, d, e)
REAL = {"a", "b", "f", "g"}
mat_files = [f for f in mat_files if os.path.basename(f)[-5] in REAL]
print("Found:", [os.path.basename(f) for f in mat_files])

## 3. Load one subject and look inside

Each .mat file holds three things:
- `cnt`: the continuous recording, shape (time, 59). Stored as 16-bit integers in units of
  0.1 microvolt, so we multiply by 0.1 to get microvolts.
- `mrk.pos`: the sample index where each cue appeared (200 cues). `mrk.y`: the class, -1 or +1.
- `nfo`: metadata: sampling rate `fs`, channel names `clab`, class names, electrode x/y positions.

In [ ]:
def load_subject(mat_path):
    m = loadmat(mat_path, squeeze_me=True, struct_as_record=False)
    cnt = 0.1 * m["cnt"].astype(np.float64)              # (time, channels) in microvolts
    pos = np.atleast_1d(m["mrk"].pos).astype(int)         # cue sample indices
    y = np.atleast_1d(m["mrk"].y).astype(int)             # -1 / +1
    nfo = m["nfo"]
    info = {
        "fs": int(nfo.fs),
        "clab": [str(c) for c in np.atleast_1d(nfo.clab)],
        "classes": [str(c) for c in np.atleast_1d(nfo.classes)],
        "xpos": np.asarray(nfo.xpos, dtype=float),
        "ypos": np.asarray(nfo.ypos, dtype=float),
    }
    return cnt, pos, y, info

cnt, pos, y, info = load_subject(mat_files[0])
print("Continuous EEG shape (time, channels):", cnt.shape)
print("Sampling rate:", info["fs"], "Hz  ->", cnt.shape[0] / info["fs"] / 60, "minutes of recording")
print("Number of cues:", len(pos), "| classes:", info["classes"], "| labels:", dict(zip(*np.unique(y, return_counts=True))))
print("First 10 channel names:", info["clab"][:10])

In [ ]:
# Plot 5 seconds of 5 channels, raw. This is what EEG looks like: small, noisy waves.
t = np.arange(0, 5 * info["fs"]) / info["fs"]
plt.figure(figsize=(12, 4))
for k in range(5):
    plt.plot(t, cnt[:len(t), k] + 40 * k, lw=0.7, label=info["clab"][k])   # offset each channel
plt.xlabel("seconds"); plt.ylabel("microvolts (offset)"); plt.title("Raw EEG, first 5 s"); plt.legend(loc="right")
plt.show()

## 4. Preprocess: band-pass filter, then cut into trials

Motor imagery shows up in two rhythms: **mu (8-13 Hz)** and **beta (13-30 Hz)**. Imagining a
hand movement *reduces* their power over the opposite side of the motor cortex. So we keep
only 8-30 Hz with a band-pass filter, which also removes slow drifts and most muscle noise.

Then we cut the continuous signal into **trials**: for each cue, take the window from 0.5 s
to 3.5 s after the cue (3 s = 300 samples at 100 Hz). The result is our tensor:
`(trials, channels, time)`.

Finally we standardise each trial (subtract the mean, divide by the standard deviation) so
one loud trial can't dominate the decomposition.

In [ ]:
def bandpass(data, fs, low=8.0, high=30.0, order=4):
    """Zero-phase Butterworth band-pass along the time axis (axis 0)."""
    b, a = butter(order, [low / (fs / 2), high / (fs / 2)], btype="band")
    return filtfilt(b, a, data, axis=0)

def make_trials(cnt, pos, fs, t_start=0.5, t_end=3.5):
    s0, s1 = int(t_start * fs), int(t_end * fs)                # 50 and 350 samples
    trials = np.stack([cnt[p + s0:p + s1].T for p in pos])     # (n_trials, channels, time)
    return trials

cnt_f = bandpass(cnt, info["fs"])
X = make_trials(cnt_f, pos, info["fs"])
X = (X - X.mean(axis=(1, 2), keepdims=True)) / X.std(axis=(1, 2), keepdims=True)
labels = (y > 0).astype(int)                                   # -1/+1 -> 0/1
print("Trial tensor shape (trials, channels, time):", X.shape)

## 5. CP decomposition: what does it do?

A rank-R CP model says the tensor is approximately

```
X[trial, channel, time]  ~  sum over r=1..R of  C[trial, r] * A[channel, r] * B[time, r]
```

- `A` (59 x R): R spatial maps. Which electrodes each pattern lives on.
- `B` (300 x R): R time courses. How each pattern rises and falls after the cue.
- `C` (200 x R): R loadings per trial. **These are our features.**

Alternating least squares finds A, B, C that make the approximation as close as possible.
`tensorly.decomposition.parafac` does the work.

In [ ]:
RANK = 4

def fit_cp(X_train, rank=RANK, seed=SEED):
    weights, (C, A, B) = parafac(tl.tensor(X_train), rank=rank, init="random",
                                 random_state=seed, n_iter_max=300, tol=1e-7, normalize_factors=True)
    # fold the per-component weights into the trial factor so A and B stay unit-norm
    return A, B, C * weights

A, B, C = fit_cp(X)
print("Spatial maps A:", A.shape, "| time courses B:", B.shape, "| trial loadings C:", C.shape)

# How much of the data does the rank-4 model explain?
recon = tl.cp_to_tensor((np.ones(RANK), [C, A, B]))
explained = 1 - np.sum((X - recon) ** 2) / np.sum(X ** 2)
print(f"Variance explained by {RANK} components: {explained*100:.1f}%")

### Look at the four patterns

Top row: the spatial map of each component drawn on the electrode layout (red = positive,
blue = negative). Bottom row: its time course across the 3-second window.

In [ ]:
def plot_components(A, B, info, fs, title=""):
    R = A.shape[1]
    fig, axes = plt.subplots(2, R, figsize=(3.2 * R, 6))
    for r in range(R):
        ax = axes[0, r]
        v = np.abs(A[:, r]).max()
        sc = ax.scatter(info["xpos"], info["ypos"], c=A[:, r], cmap="RdBu_r", vmin=-v, vmax=v, s=60)
        ax.set_title(f"component {r+1}: where"); ax.set_aspect("equal"); ax.axis("off")
        ax = axes[1, r]
        ax.plot(np.arange(B.shape[0]) / fs + 0.5, B[:, r], lw=0.8)
        ax.set_title("when"); ax.set_xlabel("s after cue")
    fig.suptitle(title); plt.tight_layout(); plt.show()

plot_components(A, B, info, info["fs"], title=f"CP components, subject {os.path.basename(mat_files[0])[-5]}")

## 6. Decoding: do the 4 loadings separate the two classes?

Honest protocol, 10-fold cross-validation:
1. Fit CP on the **training** trials only -> spatial maps A and time courses B.
2. For each **test** trial, find the 4 loadings that best reconstruct it using the fixed A
   and B (a least-squares projection, no peeking at its label).
3. Train a **Linear Discriminant Analysis** classifier on the training loadings, test it on
   the test loadings.

LDA is the standard simple classifier in BCI work: it draws one straight boundary between the
two classes in the 4-dimensional loading space.

A first-order comparison, a plain **log band-power + LDA** decoder on the same trials, tells
us whether 4 CP numbers compete with 59 hand-crafted power features.

In [ ]:
def project_trials(X_new, A, B):
    """Least-squares loadings of new trials on fixed spatial (A) and temporal (B) factors."""
    KR = khatri_rao([B, A])                               # (time*channels, R)
    pinv = np.linalg.pinv(KR)                             # (R, time*channels)
    # vectorise each trial as (channels, time) -> column-major to match khatri_rao([B, A])
    V = X_new.reshape(X_new.shape[0], -1, order="F")      # (n, channels*time)
    return V @ pinv.T                                     # (n, R)

def cv_decode(X, labels, rank=RANK, n_splits=10, seed=SEED):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    cp_acc, bp_acc = [], []
    for tr, te in skf.split(X, labels):
        # --- CP features ---
        A, B, C_tr = fit_cp(X[tr], rank=rank, seed=seed)
        C_te = project_trials(X[te], A, B)
        clf = make_pipeline(StandardScaler(), LinearDiscriminantAnalysis())
        clf.fit(C_tr, labels[tr]);  cp_acc.append(clf.score(C_te, labels[te]))
        # --- baseline: log band power per channel ---
        P_tr = np.log(X[tr].var(axis=2));  P_te = np.log(X[te].var(axis=2))
        clf2 = make_pipeline(StandardScaler(), LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto"))
        clf2.fit(P_tr, labels[tr]);  bp_acc.append(clf2.score(P_te, labels[te]))
    return np.mean(cp_acc), np.std(cp_acc), np.mean(bp_acc), np.std(bp_acc)

cp_m, cp_s, bp_m, bp_s = cv_decode(X, labels)
print(f"Subject {os.path.basename(mat_files[0])[-5]}")
print(f"  CP rank-{RANK} loadings + LDA : {cp_m*100:5.1f}% (+/- {cp_s*100:.1f})")
print(f"  log band-power (59) + LDA  : {bp_m*100:5.1f}% (+/- {bp_s*100:.1f})")
print(f"  chance                     : 50.0%")

## 7. All four real subjects

BCI performance varies a lot between people. We repeat the whole pipeline per subject.

In [ ]:
per_subject = {}
for f in mat_files:
    sid = os.path.basename(f)[-5]
    cnt, pos, y, info = load_subject(f)
    Xs = make_trials(bandpass(cnt, info["fs"]), pos, info["fs"])
    Xs = (Xs - Xs.mean(axis=(1, 2), keepdims=True)) / Xs.std(axis=(1, 2), keepdims=True)
    ls = (y > 0).astype(int)
    cp_m, cp_s, bp_m, bp_s = cv_decode(Xs, ls)
    per_subject[sid] = {"classes": info["classes"], "cp_acc": cp_m, "cp_std": cp_s, "bp_acc": bp_m, "bp_std": bp_s}
    print(f"subject {sid} ({' vs '.join(info['classes'])}): CP {cp_m*100:5.1f}% | band-power {bp_m*100:5.1f}%")

mean_cp = np.mean([v["cp_acc"] for v in per_subject.values()])
mean_bp = np.mean([v["bp_acc"] for v in per_subject.values()])
print(f"\nMean over subjects: CP rank-{RANK} {mean_cp*100:.1f}% | band-power {mean_bp*100:.1f}%")

In [ ]:
sids = list(per_subject)
xpos_ = np.arange(len(sids))
plt.figure(figsize=(7, 4))
plt.bar(xpos_ - 0.2, [per_subject[s]["cp_acc"] * 100 for s in sids], 0.4, label=f"CP rank-{RANK} + LDA",
        yerr=[per_subject[s]["cp_std"] * 100 for s in sids], capsize=3)
plt.bar(xpos_ + 0.2, [per_subject[s]["bp_acc"] * 100 for s in sids], 0.4, label="band-power + LDA",
        yerr=[per_subject[s]["bp_std"] * 100 for s in sids], capsize=3)
plt.axhline(50, color="gray", ls="--", lw=1, label="chance")
plt.xticks(xpos_, [f"subject {s}" for s in sids]); plt.ylabel("10-fold CV accuracy (%)"); plt.ylim(0, 100)
plt.title("Single-trial motor imagery decoding"); plt.legend(); plt.show()

## 8. How many components do we need?

Rank is the one knob in CP. Sweep it on one subject to see where accuracy saturates.

In [ ]:
cnt, pos, y, info = load_subject(mat_files[0])
X = make_trials(bandpass(cnt, info["fs"]), pos, info["fs"])
X = (X - X.mean(axis=(1, 2), keepdims=True)) / X.std(axis=(1, 2), keepdims=True)
labels = (y > 0).astype(int)

ranks = [1, 2, 3, 4, 6, 8]
sweep = []
for r in ranks:
    m, s, _, _ = cv_decode(X, labels, rank=r, n_splits=5)
    sweep.append(m); print(f"rank {r}: {m*100:5.1f}%")
plt.figure(figsize=(6, 4)); plt.plot(ranks, np.array(sweep) * 100, marker="o")
plt.axhline(50, color="gray", ls="--", lw=1); plt.xlabel("CP rank"); plt.ylabel("accuracy (%)")
plt.title(f"Accuracy vs number of components (subject {os.path.basename(mat_files[0])[-5]})"); plt.show()

## 9. Save results

In [ ]:
summary = {
    "dataset": "BCI Competition IV, data set 1, 100 Hz calibration, real subjects a/b/f/g",
    "channels": 59, "band_hz": [8, 30], "window_s": [0.5, 3.5], "cp_rank": RANK,
    "per_subject": {s: {k: (round(float(v), 3) if isinstance(v, (float, np.floating)) else v)
                        for k, v in d.items()} for s, d in per_subject.items()},
    "mean_cp_accuracy_pct": round(mean_cp * 100, 1),
    "mean_bandpower_accuracy_pct": round(mean_bp * 100, 1),
    "rank_sweep_subject_a": dict(zip(ranks, [round(v * 100, 1) for v in sweep])),
}
with open("results_eeg_cp.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))